In [12]:
"""
=============================================================
  DATA STORYTELLING: SURVIVAL ON THE TITANIC
=============================================================
  Story Arcs:
    1. Class vs. Conscience   —  Socioeconomic inequality
    2. Women & Children First —  Was the rule really followed?
    3. What's in a Name?      —  Titles & social rank
    4. Would YOU survive?     —  Interactive terminal predictor
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import re
import os
import sys

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# ── Palette
SURVIVED_COLOR   = "#2ecc71"
PERISHED_COLOR   = "#e74c3c"
CLASS1_COLOR     = "#f39c12"
CLASS2_COLOR     = "#3498db"
CLASS3_COLOR     = "#9b59b6"
BG_COLOR         = "#1a1a2e"
TEXT_COLOR       = "#eaeaea"
GRID_COLOR       = "#2d2d4e"

plt.rcParams.update({
    "figure.facecolor":  BG_COLOR,
    "axes.facecolor":    BG_COLOR,
    "axes.edgecolor":    GRID_COLOR,
    "axes.labelcolor":   TEXT_COLOR,
    "xtick.color":       TEXT_COLOR,
    "ytick.color":       TEXT_COLOR,
    "text.color":        TEXT_COLOR,
    "grid.color":        GRID_COLOR,
    "grid.linestyle":    "--",
    "grid.alpha":        0.4,
    "font.family":       "DejaVu Sans",
})

OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:
# 1. LOAD & CLEAN DATA
def load_data(path="datasets\Day 1\train.csv"):
    if not os.path.exists(path):
        print(f"\n  '{path}' not found in the current directory.")
        print("   Download train.csv from: https://www.kaggle.com/competitions/titanic/data")
        print("   Then place it in the same folder as this script.\n")
        sys.exit(1)
    return pd.read_csv(path)

def extract_title(name):
    match = re.search(r",\s*([^.]+)\.", name)
    if not match:
        return "Other"
    title = match.group(1).strip()
    noble    = {"Don","Sir","Jonkheer","the Countess","Lady","Dona"}
    military = {"Capt","Col","Major"}
    if title in ["Mr"]:            return "Mr"
    if title in ["Miss","Mlle","Ms"]: return "Miss"
    if title in ["Mrs","Mme"]:     return "Mrs"
    if title == "Master":          return "Master"
    if title in ["Dr","Rev"]:      return "Professional"
    if title in noble:             return "Noble"
    if title in military:         return "Military"
    return "Other"

def preprocess(df):
    df = df.copy()
    df["Age"].fillna(df["Age"].median(), inplace=True)
    df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)
    df["Fare"].fillna(df["Fare"].median(), inplace=True)
    df["Title"]       = df["Name"].apply(extract_title)
    df["FamilySize"]  = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"]     = (df["FamilySize"] == 1).astype(int)
    df["AgeGroup"]    = pd.cut(df["Age"],
                                bins=[0,12,18,35,60,100],
                                labels=["Child","Teen","Young Adult","Adult","Senior"])
    return df

In [14]:
# 2. STORY ARC 1 — CLASS VS. CONSCIENCE
def plot_class_survival(df):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("STORY ARC 1 — CLASS VS. CONSCIENCE\nDid your ticket price determine your fate?",
                 fontsize=16, fontweight="bold", color=TEXT_COLOR, y=1.02)

    # ── (a) Survival rate by class
    ax = axes[0]
    sr = df.groupby("Pclass")["Survived"].mean() * 100
    colors = [CLASS1_COLOR, CLASS2_COLOR, CLASS3_COLOR]
    bars = ax.bar(["1st Class","2nd Class","3rd Class"], sr.values, color=colors,
                  edgecolor="white", linewidth=0.5, width=0.55)
    for bar, val in zip(bars, sr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", color=TEXT_COLOR)
    ax.set_ylim(0, 85)
    ax.set_title("Survival Rate by Class", fontweight="bold")
    ax.set_ylabel("Survival Rate (%)")
    ax.grid(axis="y")

    # ── (b) Fare distribution vs survival
    ax = axes[1]
    survived = df[df["Survived"]==1]["Fare"].clip(upper=300)
    perished = df[df["Survived"]==0]["Fare"].clip(upper=300)
    ax.hist(perished, bins=30, color=PERISHED_COLOR, alpha=0.7, label="Perished", density=True)
    ax.hist(survived, bins=30, color=SURVIVED_COLOR, alpha=0.7, label="Survived", density=True)
    ax.axvline(survived.median(), color=SURVIVED_COLOR, linestyle="--", linewidth=1.5,
               label=f"Survived median: £{survived.median():.0f}")
    ax.axvline(perished.median(), color=PERISHED_COLOR, linestyle="--", linewidth=1.5,
               label=f"Perished median: £{perished.median():.0f}")
    ax.set_title("Fare Distribution vs Survival", fontweight="bold")
    ax.set_xlabel("Fare (£) — capped at 300")
    ax.legend(fontsize=8)
    ax.grid(axis="y")

    # ── (c) Stacked count: class + survival
    ax = axes[2]
    ct = df.groupby(["Pclass","Survived"]).size().unstack()
    ct.columns = ["Perished","Survived"]
    ct.index   = ["1st Class","2nd Class","3rd Class"]
    ct[["Perished","Survived"]].plot(
        kind="bar", stacked=True, ax=ax,
        color=[PERISHED_COLOR, SURVIVED_COLOR],
        edgecolor="white", linewidth=0.4, width=0.55
    )
    ax.set_title("Passenger Count — Class & Outcome", fontweight="bold")
    ax.set_ylabel("Number of Passengers")
    ax.set_xlabel("")
    plt.setp(ax.get_xticklabels(), rotation=0)
    ax.legend(fontsize=9)
    ax.grid(axis="y")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "arc1_class_vs_conscience.png")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close()
    print(f" Saved: arc1_class_vs_conscience.png")

In [15]:
# 3. STORY ARC 2 — WOMEN & CHILDREN FIRST
def plot_women_children(df):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('STORY ARC 2 — "WOMEN & CHILDREN FIRST"\nWas the maritime rule actually followed?',
                 fontsize=16, fontweight="bold", color=TEXT_COLOR, y=1.02)

    # ── (a) Survival by gender
    ax = axes[0]
    gsr = df.groupby("Sex")["Survived"].mean() * 100
    colors = ["#e91e8c","#1e90ff"]
    bars = ax.bar(["Female","Male"], [gsr["female"], gsr["male"]],
                  color=colors, edgecolor="white", linewidth=0.5, width=0.45)
    for bar, val in zip(bars, [gsr["female"], gsr["male"]]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.1f}%", ha="center", fontweight="bold", color=TEXT_COLOR)
    ax.set_ylim(0, 100)
    ax.set_title("Survival Rate by Gender", fontweight="bold")
    ax.set_ylabel("Survival Rate (%)")
    ax.axhline(50, color="white", linestyle=":", alpha=0.4)
    ax.grid(axis="y")

    # ── (b) Age group survival
    ax = axes[1]
    age_sr = df.groupby("AgeGroup", observed=True)["Survived"].mean() * 100
    bar_colors = [CLASS1_COLOR, CLASS2_COLOR, SURVIVED_COLOR, CLASS3_COLOR, PERISHED_COLOR]
    bars = ax.bar(age_sr.index.astype(str), age_sr.values,
                  color=bar_colors, edgecolor="white", linewidth=0.5, width=0.55)
    for bar, val in zip(bars, age_sr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold", color=TEXT_COLOR)
    ax.set_ylim(0, 75)
    ax.set_title("Survival Rate by Age Group", fontweight="bold")
    ax.set_ylabel("Survival Rate (%)")
    ax.axhline(50, color="white", linestyle=":", alpha=0.4)
    ax.grid(axis="y")

    # ── (c) Heatmap: Gender × Class × Survival
    ax = axes[2]
    pivot = df.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="mean") * 100
    pivot.index = ["Female","Male"]
    pivot.columns = ["1st Class","2nd Class","3rd Class"]
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".1f", cmap="RdYlGn",
                linewidths=0.5, linecolor=BG_COLOR,
                annot_kws={"size":13,"weight":"bold"},
                cbar_kws={"label":"Survival Rate (%)"})
    ax.set_title("Survival % — Gender × Class", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "arc2_women_children_first.png")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close()
    print(f" Saved: arc2_women_children_first.png")

In [16]:
# 4. STORY ARC 3 — WHAT'S IN A NAME? (TITLES)
def plot_titles(df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("STORY ARC 3 — WHAT'S IN A NAME?\nDid your social title seal your fate?",
                 fontsize=16, fontweight="bold", color=TEXT_COLOR, y=1.02)

    title_order = ["Mr","Mrs","Miss","Master","Professional","Noble","Military","Other"]
    palette = {
        "Mr":           PERISHED_COLOR,
        "Mrs":          "#e91e8c",
        "Miss":         "#ff69b4",
        "Master":       CLASS1_COLOR,
        "Professional": CLASS2_COLOR,
        "Noble":        "#f1c40f",
        "Military":     "#7f8c8d",
        "Other":        "#95a5a6",
    }

    # ── (a) Survival rate by title
    ax = axes[0]
    tsr = (df.groupby("Title")["Survived"].mean() * 100).reindex(title_order).dropna()
    bar_colors = [palette.get(t, "#aaa") for t in tsr.index]
    bars = ax.barh(tsr.index, tsr.values, color=bar_colors, edgecolor="white", linewidth=0.4)
    for bar, val in zip(bars, tsr.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}%", va="center", fontsize=9, fontweight="bold", color=TEXT_COLOR)
    ax.set_xlim(0, 105)
    ax.set_title("Survival Rate by Title", fontweight="bold")
    ax.set_xlabel("Survival Rate (%)")
    ax.axvline(50, color="white", linestyle=":", alpha=0.4)
    ax.grid(axis="x")

    # ── (b) Count: who was on board?
    ax = axes[1]
    tc = df["Title"].value_counts().reindex(title_order).dropna()
    bar_colors2 = [palette.get(t, "#aaa") for t in tc.index]
    bars = ax.bar(tc.index, tc.values, color=bar_colors2, edgecolor="white", linewidth=0.4, width=0.6)
    for bar, val in zip(bars, tc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(val), ha="center", fontsize=9, fontweight="bold", color=TEXT_COLOR)
    ax.set_title("Passenger Count by Title", fontweight="bold")
    ax.set_ylabel("Number of Passengers")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    ax.grid(axis="y")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "arc3_titles_social_rank.png")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close()
    print(f" Saved: arc3_titles_social_rank.png")

In [17]:
# 5. TRAIN MODEL (for predictor)
def train_model(df):
    features = ["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked","FamilySize","IsAlone"]
    le_sex      = LabelEncoder()
    le_embarked = LabelEncoder()
    X = df[features].copy()
    X["Sex"]      = le_sex.fit_transform(X["Sex"])
    X["Embarked"] = le_embarked.fit_transform(X["Embarked"])
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test)) * 100
    print(f"\n Model trained — Accuracy on held-out test set: {acc:.1f}%")
    return model, le_sex, le_embarked, acc

In [18]:
# 6. INTERACTIVE TERMINAL PREDICTOR
def get_int(prompt, valid_range):
    while True:
        try:
            val = int(input(prompt))
            if val in valid_range:
                return val
            print(f"Please enter one of: {list(valid_range)}")
        except ValueError:
            print("Please enter a valid number.")

def get_float(prompt, low=0, high=9999):
    while True:
        try:
            val = float(input(prompt))
            if low <= val <= high:
                return val
            print(f"Please enter a value between {low} and {high}.")
        except ValueError:
            print("Please enter a valid number.")

def get_choice(prompt, choices):
    choices_lower = [c.lower() for c in choices]
    while True:
        val = input(prompt).strip().lower()
        if val in choices_lower:
            return choices[choices_lower.index(val)]
        print(f"Please enter one of: {choices}")

def run_predictor(model, le_sex, le_embarked, model_accuracy):
    BANNER = f"""
══════════════════════════════════════════════════════════
            TITANIC SURVIVAL PREDICTOR
        April 15, 1912 — North Atlantic Ocean
        Model Accuracy: {model_accuracy:.1f}%
══════════════════════════════════════════════════════════
  Answer a few questions to find out if you would have
  survived the sinking of RMS Titanic.
"""
    print(BANNER)

    while True:
        print("─" * 58)
        # ── Inputs
        pclass    = get_int(
            "  [1] Ticket Class  (1 = First / 2 = Second / 3 = Third): ",
            range(1, 4)
        )
        sex       = get_choice(
            "  [2] Gender        (male / female): ",
            ["male","female"]
        )
        age       = get_float(
            "  [3] Your Age      (0–100): ",
            0, 100
        )
        sibsp     = get_int(
            "  [4] Siblings / Spouses aboard (0–8): ",
            range(0, 9)
        )
        parch     = get_int(
            "  [5] Parents / Children aboard (0–6): ",
            range(0, 7)
        )
        fare      = get_float(
            "  [6] Ticket Fare in £ (e.g. 7.25 for 3rd, 30 for 2nd, 100+ for 1st): ",
            0, 600
        )
        embarked  = get_choice(
            "  [7] Port of Embarkation  (S = Southampton / C = Cherbourg / Q = Queenstown): ",
            ["S","C","Q"]
        )

        # ── Feature engineering (mirror training)
        family_size = sibsp + parch + 1
        is_alone    = 1 if family_size == 1 else 0
        sex_enc     = le_sex.transform([sex])[0]
        emb_enc     = le_embarked.transform([embarked])[0]

        X_input = pd.DataFrame([{
            "Pclass":     pclass,
            "Sex":        sex_enc,
            "Age":        age,
            "SibSp":      sibsp,
            "Parch":      parch,
            "Fare":       fare,
            "Embarked":   emb_enc,
            "FamilySize": family_size,
            "IsAlone":    is_alone,
        }])

        prob       = model.predict_proba(X_input)[0][1]
        survived   = prob >= 0.5
        confidence = prob * 100 if survived else (1 - prob) * 100

        # ── Context clues
        context_msgs = []
        if sex == "female":
            context_msgs.append("Being female significantly improved survival odds.")
        if pclass == 1:
            context_msgs.append("1st class passengers had much better access to lifeboats.")
        elif pclass == 3:
            context_msgs.append("3rd class passengers faced locked gates and distance to decks.")
        if age <= 12:
            context_msgs.append("Children were generally prioritised in the evacuation.")
        if is_alone:
            context_msgs.append("Solo travellers had mixed outcomes — no family coordination.")
        if family_size > 4:
            context_msgs.append("Large families often struggled to evacuate together.")
        if fare > 50:
            context_msgs.append("Higher fare suggests better cabin location (closer to boat deck).")

        # ── Result display
        print("\n" + "═" * 58)
        if survived:
            print(f"""
      VERDICT: YOU WOULD HAVE SURVIVED
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Survival Probability : {prob*100:.1f}%
  Confidence           : {confidence:.1f}%

      You made it onto a lifeboat and were rescued
      by the RMS Carpathia at dawn.
""")
        else:
            print(f"""
      VERDICT: YOU WOULD NOT HAVE SURVIVED
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Survival Probability : {prob*100:.1f}%
  Confidence           : {confidence:.1f}%

      Unfortunately, you perished in the
      freezing North Atlantic waters.
""")

        if context_msgs:
            print("Key factors in your outcome:")
            for msg in context_msgs:
                print(f"       • {msg}")

        print("═" * 58)

        again = input("\n  Would you like to try different details? (yes / no): ").strip().lower()
        if again not in ("yes","y"):
            print("\n  Thanks for exploring the Titanic story. Fair winds!\n")
            break
        print()

In [19]:
# 7. MAIN

import os
BASE = os.path.dirname(os.getcwd())
TRAIN_PATH = os.path.join(BASE, "datasets", "Day 1", "train.csv")

print("\n" + "═"*60)
print("  DATA STORYTELLING: SURVIVAL ON THE TITANIC")
print("═"*60)

# ── Load & preprocess
print("\nLoading dataset...")
df = load_data(TRAIN_PATH)
df = preprocess(df)
print(f" {len(df)} passengers loaded | {df['Survived'].sum()} survived | "
      f"{len(df) - df['Survived'].sum()} perished")

# ── Visualisations
print("\nGenerating story visualisations...")
plot_class_survival(df)
plot_women_children(df)
plot_titles(df)

# ── Train model
print("\nTraining survival model...")
model, le_sex, le_embarked, acc = train_model(df)

print("\nAll 3 story arc charts saved to the outputs folder.")
print("    Open them to explore the data narratives!\n")

# ── Interactive predictor
launch = input("Launch the interactive Titanic Survival Predictor? (yes / no): ").strip().lower()
if launch in ("yes", "y"):
    run_predictor(model, le_sex, le_embarked, acc)


════════════════════════════════════════════════════════════
  DATA STORYTELLING: SURVIVAL ON THE TITANIC
════════════════════════════════════════════════════════════

Loading dataset...
 891 passengers loaded | 342 survived | 549 perished

Generating story visualisations...
 Saved: arc1_class_vs_conscience.png
 Saved: arc2_women_children_first.png
 Saved: arc3_titles_social_rank.png

Training survival model...

 Model trained — Accuracy on held-out test set: 79.9%

All 3 story arc charts saved to the outputs folder.
    Open them to explore the data narratives!


══════════════════════════════════════════════════════════
            TITANIC SURVIVAL PREDICTOR
        April 15, 1912 — North Atlantic Ocean
        Model Accuracy: 79.9%
══════════════════════════════════════════════════════════
  Answer a few questions to find out if you would have
  survived the sinking of RMS Titanic.

──────────────────────────────────────────────────────────

════════════════════════════════════════